This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [14]:
!pip3 install -r requirements.txt

  Using cached langchain-0.0.305-py3-none-any.whl (1.8 MB)
  Using cached pytest-8.3.2-py3-none-any.whl (341 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Using cached jupyter-1.0.0-py2.py3-none-any.whl (2.7 kB)
     |████████████████████████████████| 80 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 15.3 MB/s eta 0:00:01
  Using cached numexpr-2.10.1-cp39-cp39-macosx_11_0_arm64.whl (130 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached aiohttp-3.10.5-cp39-cp39-macosx_11_0_arm64.whl (389 kB)
  Using cached ipykernel-6.29.5-py3-none-any.whl (117 kB)
     |████████████████████████████████| 123 kB 76.6 MB/s eta 0:00:01
  Using cached jupyter_console-6.6.3-py3-no

In [18]:
from langchain_community.llms import OpenAI
from langchain.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain.vectorstores import Chroma
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field, NonNegativeInt
from comet_ml import Experiment
from fastapi.encoders import jsonable_encoder
import torch
from diffusers import StableDiffusionPipeline
from langchain.output_parsers import PydanticOutputParser
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain.chains import RetrievalQA
from langchain.chains.question_answering import load_qa_chain
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import PIL
import os
import gc
from dotenv import load_dotenv
import pandas as pd
from typing import List

# Load environment variables
load_dotenv('my_config.env')

# API configuration
API_KEY = os.getenv('API_KEY')
openai_api_key = os.getenv("OPENAI_API_KEY")
COMET_API_KEY = os.getenv("COMET_API_KEY")


# Initialize the experiment
experiment = Experiment(
    api_key=COMET_API_KEY,
    project_name="real-estate-agent",
    workspace="polarbeargo",
    log_code=True,
)

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : net_tuna_7141
COMET INFO:     url                   : https://www.comet.com/polarbeargo/real-estate-agent/af898b218ca04cebb256b8afc01dd31d
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (3.99 KB)
COMET INFO:     installed packages       : 1
COMET INFO:     notebook                 : 1
COMET INFO:     source_code              : 1
COMET INFO: 
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged b

- Define the prompt for generating synthetic real estate data (images and text)

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [19]:
instruction = """
Generate eighteen realistic real estate listings from a wide range of neighborhoods.
"""

template = \
"""
Here is the template of real estate listing:

Neighborhood: Mountain View
Price: $650,000
Bedrooms: 5
Bathrooms: 4
House Size: 3000 sqft
Description: Spacious family home with breathtaking views of the mountains and a large backyard for outdoor entertaining.
Neighborhood Description: Mountain View is known for its scenic landscape and outdoor activities, making it the ideal location for nature lovers and adventure seekers.
"""
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, api_key=openai_api_key, max_tokens = 500)
image_model = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")

# For Apple Silicon (M1/M2) replace mps to device if executing on other devices
image_model.to("mps")

image_dir = "generated_images"
os.makedirs(image_dir, exist_ok=True)

# Define the Listing data model
class Listing(BaseModel):
    neighborhood: str = Field(description="The neighborhood where the property is located.")
    price: NonNegativeInt = Field(description="The price of the property in USD.")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms in the property.")
    bathrooms: NonNegativeInt = Field(description="The number of bathrooms in the property.")
    house_size: NonNegativeInt = Field(description="The size of the property in square feet.")
    description: str = Field(description="A brief description of the property.")
    neighborhood_description: str = Field(description="A description of the neighborhood where the property is located.")

class Listings(BaseModel):
    listing: List[Listing] = Field(description="List of available real estate listings.")
    

def create_listing_prompt(listing: Listing) -> str:
    return f"""
    Neighborhood: {listing.neighborhood}
    Price: ${listing.price}
    Bedrooms: {listing.bedrooms}
    Bathrooms: {listing.bathrooms}
    House Size: {listing.house_size} sqft
    Description: {listing.description}
    Neighborhood Description: {listing.neighborhood_description}
    """

# Define few-shot examples
examples = [
    {
        "question": "Generate a listing for a 3-bedroom house in downtown.",
        "answer": Listing(
            neighborhood="Downtown",
            price=500000,
            bedrooms=3,
            bathrooms=2,
            house_size=1500,
            description="A beautiful 3-bedroom house located in the heart of downtown with modern amenities.",
            neighborhood_description="Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks."
        )
    },
    {
        "question": "Create a listing for a luxury apartment in the suburbs.",
        "answer": Listing(
            neighborhood="Suburbia",
            price=750000,
            bedrooms=2,
            bathrooms=2,
            house_size=1200,
            description="A luxurious apartment featuring high-end finishes and spacious living areas.",
            neighborhood_description="Suburbia offers a peaceful environment with great schools and family-friendly parks."
        )
    }
]

parser = PydanticOutputParser(pydantic_object=Listings)

example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="{question}\n{answer}",
    partial_variables={"format_instructions": parser.get_format_instructions},
)

few_shot_prompt = FewShotPromptTemplate(
    examples=[{"question": ex["question"], "answer": create_listing_prompt(ex["answer"])} for ex in examples],
    example_prompt=example_prompt,
    suffix="Use these examples to generate a listing for the following question: {input}",
    input_variables=["input"],
    partial_variables={"format_instructions": parser.get_format_instructions},
)

full_prompt = few_shot_prompt.format(sample=template, input=instruction)
response = llm(full_prompt)
print(f"Raw Response: {response}")  # Debugging line to check the response format

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Raw Response: 
1. Neighborhood: Beachfront
Price: $1,200,000
Bedrooms: 4
Bathrooms: 3
House Size: 2500 sqft
Description: Enjoy stunning ocean views from this spacious 4-bedroom beachfront home.
Neighborhood Description: Beachfront is a highly sought-after neighborhood known for its beautiful beaches and upscale living.

2. Neighborhood: Historic District
Price: $850,000
Bedrooms: 3
Bathrooms: 2
House Size: 1800 sqft
Description: Step back in time with this charming 3-bedroom home located in the historic district.
Neighborhood Description: The historic district is filled with quaint homes, tree-lined streets, and rich history.

3. Neighborhood: Mountain View
Price: $650,000
Bedrooms: 3
Bathrooms: 2
House Size: 2000 sqft
Description: Embrace the mountain lifestyle in this spacious 3-bedroom home with breathtaking views.
Neighborhood Description: Mountain View offers a peaceful and scenic setting, perfect for outdoor enthusiasts.

4. Neighborhood: City Center
Price: $900,000
Bedrooms: 2
B

In [20]:
# Split the string into individual listings
listings = response.strip().split('\n\n')
print(listings)  # Debugging line to check the listings format
data = []

for listing in listings:

    # Remove the integer before 'Neighborhood'
    listing = listing.split('. ', 1)[1]

    # Split the listing into lines
    lines = listing.split('\n')
    listing_data = {}
    
    for line in lines:
        # Check if the line contains a colon
        if ': ' in line:
            # Split on the first colon
            key, value = line.split(': ', 1)  
            listing_data[key.strip()] = value.strip()
    
    data.append(listing_data)

df = pd.DataFrame(data)

df.rename(columns={
    'Neighborhood': 'neighborhood',
    'Price': 'price',
    'Bedrooms': 'bedrooms',
    'Bathrooms': 'bathrooms',
    'House Size': 'house_size',
    'Description': 'description',
    'Neighborhood Description': 'neighborhood_description'
}, inplace=True)

# Convert price to integer and house_size to integer (removing ' sqft')
df['price'] = df['price'].replace({'\$': '', ',': ''}, regex=True).astype(int)
df['house_size'] = df['house_size'].replace({' sqft': ''}, regex=True).fillna(0).astype(int)
df


['1. Neighborhood: Beachfront\nPrice: $1,200,000\nBedrooms: 4\nBathrooms: 3\nHouse Size: 2500 sqft\nDescription: Enjoy stunning ocean views from this spacious 4-bedroom beachfront home.\nNeighborhood Description: Beachfront is a highly sought-after neighborhood known for its beautiful beaches and upscale living.', '2. Neighborhood: Historic District\nPrice: $850,000\nBedrooms: 3\nBathrooms: 2\nHouse Size: 1800 sqft\nDescription: Step back in time with this charming 3-bedroom home located in the historic district.\nNeighborhood Description: The historic district is filled with quaint homes, tree-lined streets, and rich history.', '3. Neighborhood: Mountain View\nPrice: $650,000\nBedrooms: 3\nBathrooms: 2\nHouse Size: 2000 sqft\nDescription: Embrace the mountain lifestyle in this spacious 3-bedroom home with breathtaking views.\nNeighborhood Description: Mountain View offers a peaceful and scenic setting, perfect for outdoor enthusiasts.', '4. Neighborhood: City Center\nPrice: $900,000\n

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Beachfront,1200000,4,3,2500,Enjoy stunning ocean views from this spacious ...,Beachfront is a highly sought-after neighborho...
1,Historic District,850000,3,2,1800,Step back in time with this charming 3-bedroom...,The historic district is filled with quaint ho...
2,Mountain View,650000,3,2,2000,Embrace the mountain lifestyle in this spaciou...,Mountain View offers a peaceful and scenic set...
3,City Center,900000,2,2,1500,Experience urban living at its finest in this ...,City Center is a bustling and vibrant neighbor...
4,Lakeside,1500000,5,4,3000,This stunning 5-bedroom home boasts lakefront ...,Lakeside is a prestigious neighborhood known f...
5,Suburban Heights,700000,4,3,2500,This spacious 4-bedroom home in Suburban Heigh...,Suburban Heights offers a suburban feel with e...
6,Waterfront,2000000,6,5,4000,Live the ultimate waterfront lifestyle in this,NaN


In [21]:
df.to_csv('generated_real_estate_data.csv', index_label = 'id')

In [22]:
# Batch generation of images based on the DataFrame
def generate_images(df, batch_size=2):
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i + batch_size]
        for idx, row in batch.iterrows():
            prompt = f"A {row['bedrooms']}-bedroom house in {row['neighborhood']}. {row['neighborhood_description']}"
            image = image_model(prompt, num_inference_steps=50).images[0]
            image_path = os.path.join(image_dir, f"{row['neighborhood']}_{row['bedrooms']}_bedroom.png")
            image.save(image_path)
            print(f"Generated image saved at: {image_path}")

generate_images(df)


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Beachfront_4_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Historic District_3_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

...

Generated image saved at: generated_images/Lakeside_5_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Suburban Heights_4_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Waterfront_6_bedroom.png


##### Multimodal Vector Database Store, Embeddings Transformation and Semantic Search

- Storing Listings Into a Vector Database

In [23]:
df = pd.read_csv('generated_real_estate_data.csv')
idx = [{'id':i} for i in range(len(df.index))]
image_paths = []
images = []
texts = []
text_template = """
Neighborhood: {}
Price: {}
Bedrooms: {}
Bathrooms: {}
House Size: {}

Description: {}
Neighborhood Description: {}
"""

for i, row in df.iterrows():
    image_path = os.path.join(image_dir, f"{row['neighborhood']}_{row['bedrooms']}_bedroom.png")
    image_paths.append(image_path)
    images.append(PIL.Image.open(image_path))
    texts.append(text_template.format(row['neighborhood'], row['price'], row['bedrooms'], row['bathrooms'], row['house_size'], row['description'], row['neighborhood_description']))

In [24]:
embeddings = OpenCLIPEmbeddings()
clip_db = Chroma(collection_name="real_estate_listings", embedding_function=embeddings)

clip_db.add_texts(
    texts=texts,
    metadatas=idx
)

clip_db.add_images(
    uris=image_paths,
    metadatas=idx
)

['0ff5d469-f908-445e-b3a1-c2e3c675cbad',
 'dd11710b-c7d4-4baf-b6b7-3f75945a1f60',
 '172dbd5f-5ba0-4710-b303-ce3da72ad81e',
 '931ce3f1-e7f4-41e5-81dd-981b3a86577c',
 'd7556368-3837-42b5-8c4a-c0093cc57beb',
 '7ae200c2-25a7-46db-b338-768065df0eb0',
 'c69c6d58-c531-4dae-a7d8-3d4ff8ee3a6c']

In [26]:
questions = [   
                "How big do you want your house to be?" 
                "What are 3 most important things for you in choosing this property?", 
                "Which amenities would you like?", 
                "Which transportation options are important to you?",
                "How urban do you want your neighborhood to be?",   
            ]
answers = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

- Implementing a Retrieval-Augmented Generation (RAG) system using LangChain to perform a semantic similarity search to find vectors that are semantically similar to our query and send the text associated with the vectors to LLM for summarization.

In [38]:
query = f"""
{questions[0]} {answers[0]}
{questions[1]} {answers[1]}
{questions[2]} {answers[2]}
{questions[3]} {answers[3]}
"""

use_chain_helper = True

if use_chain_helper:
    rag = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=clip_db.as_retriever())
    print(rag.run(query))
else:
    similar_docs = clip_db.similarity_search(query, k=5)
    prompt = PromptTemplate(
        template="{query}\nContext: {context}",
        input_variables=["query", "context"],
    )
    chain = load_qa_chain(llm, prompt = prompt, chain_type="stuff")
    print(chain.run(input_documents=similar_docs, query=query))


 I am looking for a comfortable three-bedroom house with a spacious kitchen and a cozy living room. The three most important things for me in choosing this property are a quiet neighborhood, good local schools, and convenient shopping options. In terms of amenities, I would like a backyard for gardening, a two-car garage, and a modern, energy-efficient heating system. As for transportation options, I would prefer easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads. I am open to living in a suburban or urban neighborhood, as long as it meets these criteria.


- Performance Evaluation Metrics
    - With generate Real Estate Listings.
    - Define Reference Listings.
    - Calculate Evaluation Metrics.

In [ ]:
def evaluate_generated_listings(generated: str, reference: str) -> dict:
    
    # Calculate BLEU score
    bleu_score = sentence_bleu([reference.split()], generated.split())
    
    # Calculate ROUGE score
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    
    return {
        "bleu": bleu_score,
        "rouge1": scores['rouge1'].fmeasure,
        "rouge2": scores['rouge2'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure
    }

reference_listings = [
    "Neighborhood: Beachfront\nPrice: $1,200,000\nBedrooms: 4\nBathrooms: 3\nHouse Size: 3000 sqft\nDescription: A stunning beachfront property with ocean views.\nNeighborhood Description: Beachfront is known for its luxurious homes and vibrant community.",
    "Neighborhood: City Center\nPrice: $800,000\nBedrooms: 2\nBathrooms: 1\nHouse Size: 900 sqft\nDescription: A modern apartment located in the heart of the city.\nNeighborhood Description: City Center is bustling with activity and offers a variety of amenities."
]

# Evaluate the generated listings
for generated, reference in zip(response, reference_listings):
    metrics = evaluate_generated_listings(generated, reference)
    print(f"Generated Listing: {generated}")
    print(f"Evaluation Metrics: {metrics}")

- Integrate kubeflow pipelines

In [ ]:
%%writefile HomeMatch.py


- Use Gradio to create an interactive interface where we can input data related to home preferences, and the model can predict the best match for them.

In [ ]:
import gradio as gr 

In [ ]:
def generate_app( retrieve_preference):
    
    with gr.Blocks() as demo:
        
        gr.Markdown("""
        # Top 10 Real Estate Listings
        1. Filling out the form base on customer's preferences below.
        2. Click on the "Get Listings" button to fetch the top 10 real estate listings based on your input preferences.
        3. Review the listings displayed below. Each listing includes an image, price, location, and a brief description.
        4. If you want to refine your search, adjust your preferences and click "Get Listings" again.

        # EXAMPLES
        Scroll down to see a few examples of real estate listings. Click on an example to see the details.
        """)

        question_input = gr.Textbox(label="Enter your question about real estate:")
        generate_button = gr.Button("Generate Listing")
        
        output_text = gr.Textbox(label="Generated Listing")

        # Use type="pil" for PIL images
        output_image = gr.Image(label="Generated Image", type="pil")  

        generate_button.click(fn=lambda q: chain_of_thoughts([q]), inputs=question_input, outputs=output_text)

        # Button to retrieve top 5 preferences
        top_5_button = gr.Button("Get Top 5 Preferences")
        top_5_output = gr.Dataframe(headers=["Image Path", "Neighborhood"])
        
        top_5_button.click(fn=retrieve_preference, outputs=top_5_output)

    demo.launch()
    
    return demo

In [ ]:
if __name__ == "__main__":
    interface = generate_app(, )

In [2]:
# When we're done with the vector store and want to clean up
 # Delete the vector store variable if it's no longer needed
del clip_db 

# Run garbage collection to help clean up any unused objects in memory
gc.collect()
experiment.end()
interface.close()

NameError: name 'experiment' is not defined